# 3 — Tiền xử lý Hán–Việt | Phó Thanh Chủ Nữ Khoa

Notebook độc lập dùng `HanVietPreprocessor.py` trước bước dóng hàng Bertalign / Vecalign / API.

- **Chỉ sửa các đường dẫn ở Cell 1.** Chạy các cell theo thứ tự.
- **Không sao chép** TXT gốc, file Python, file JSON cấu hình hay dữ liệu từ `/kaggle/input` vào thư mục output. Module được import **tại chỗ**.
- Output chỉ gồm **kết quả xử lý**: văn bản clean (phái sinh từ bản gốc), danh sách câu/JSONL, các mục nghi vấn, thống kê và `preprocess_changes.txt`.
- Không dịch lại, không sửa nghĩa hay tự xóa chú thích / liều lượng; những nghi vấn cần con người rà soát.
- Python chuẩn, **không cần GPU và không cần pip install**.

> Kaggle: sử dụng **Add Input** để gắn dataset chứa **2 file `.txt`** và file **`HanVietPreprocessor.py`**. Nếu module vẫn nằm trong ZIP, hãy tải riêng file `.py` thành Kaggle Dataset trước; notebook không giải nén ZIP vào output.

> **Quan trọng:** Ảnh trước cho thấy bản Hán bắt đầu `帶下`, bản Việt hiển thị bắt đầu `調經`. Tiền xử lý toàn bộ không có nghĩa đã xác định chương tương ứng; trước bước voting cần chọn cùng tiểu mục.

## Cell 1 — Cấu hình đường dẫn (SỬA Ở ĐÂY)

Thay ba đường dẫn `HAN_FILE`, `VI_FILE`, `MODULE_FILE` bằng đường dẫn **thật** trong tab *Input* của Kaggle. `OUTPUT_DIR` là nơi duy nhất ghi file kết quả.

Nếu chỉ muốn chạy thử mục **經水先期** (Kinh thủy tiên kỳ), bỏ dấu `#` ở 4 biến `*_START/END` trong cell dưới và xác minh tiêu đề đúng ở **dòng riêng** của hai file. Mặc định `None` = giữ toàn bộ sách.

In [ ]:
from pathlib import Path

# ============= INPUT: SỬA 3 ĐƯỜNG DẪN NÀY =============
HAN_FILE = Path("/kaggle/input/nu-khoa-data/Pho_Thanh_Chu_Nu_Khoa.txt")
VI_FILE = Path("/kaggle/input/nu-khoa-data/nu_khoa_viet.txt")
MODULE_FILE = Path("/kaggle/input/nlp-preprocessor/HanVietPreprocessor.py")

# Nếu đã có file JSON sửa OCR/chỉnh tay được duyệt thì điền Path(...), nếu không để None.
MANUAL_CONFIG_FILE = None
# Ví dụ: MANUAL_CONFIG_FILE = Path("/kaggle/input/nu-khoa-data/manual_config.json")

# ============= OUTPUT: CHỈ KẾT QUẢ XỬ LÝ =============
OUTPUT_DIR = Path("/kaggle/working/nu_khoa_preprocessed")

# ============= CHỌN CHƯƠNG/MỤC: MẶC ĐỊNH XỬ LÝ CẢ 2 FILE =============
HAN_START = None
HAN_END   = None
VI_START  = None
VI_END    = None

# Demo một tiểu mục: thay các dòng trên bằng 4 dòng sau nếu tiêu đề thực sự có mặt.
# HAN_START = "經水先期"
# HAN_END   = "經水後期"
# VI_START  = "經水先期"
# VI_END    = "經水後期"

print("HAN_FILE :", HAN_FILE)
print("VI_FILE  :", VI_FILE)
print("MODULE   :", MODULE_FILE)
print("OUTPUT   :", OUTPUT_DIR)
print("ANCHORS  :", (HAN_START, HAN_END, VI_START, VI_END))

## Cell 2 — Tìm đường dẫn Input khi cần

Cell này **chỉ liệt kê** TXT, module `.py` hoặc `.json` trong Kaggle Input; không đọc/ghi vào output. Dựa trên danh sách để sửa Cell 1.

In [ ]:
import os

root = Path("/kaggle/input")
if root.exists():
    found = []
    for base, dirs, files in os.walk(root):
        for file_name in files:
            if file_name.lower().endswith((".txt", ".json")) or file_name == "HanVietPreprocessor.py":
                found.append(str(Path(base) / file_name))
    print(f"Tìm thấy {len(found)} file TXT/JSON hoặc HanVietPreprocessor.py:")
    for p in found[:80]:
        print(" -", p)
    if len(found) > 80:
        print(f"... còn {len(found) - 80} file khác (chỉ in 80 file đầu).")
else:
    print("Không có /kaggle/input (nếu chạy ở máy cá nhân, sửa Path tại Cell 1).")

## Cell 3 — Kiểm tra file đầu vào, không ghi file

Báo lỗi rõ nếu thiếu đường dẫn, sai đuôi file, đầu vào không đọc được dưới dạng UTF-8 hoặc output trùng với đầu vào. Module sẽ được nạp trực tiếp từ vị trí hiện có.

In [ ]:
import hashlib
import json

sources = {"Hán": HAN_FILE, "Việt": VI_FILE, "Module": MODULE_FILE}
if MANUAL_CONFIG_FILE is not None:
    sources["Cấu hình sửa tay"] = Path(MANUAL_CONFIG_FILE)

missing = [(name, path) for name, path in sources.items() if not path.is_file()]
if missing:
    for name, path in missing:
        print(f"KHÔNG TÌM THẤY {name}: {path}")
    raise FileNotFoundError("Sửa các biến đường dẫn trong Cell 1 theo danh sách ở Cell 2.")

if HAN_FILE.suffix.lower() != ".txt" or VI_FILE.suffix.lower() != ".txt":
    raise ValueError("HAN_FILE và VI_FILE phải là file .txt")
if MODULE_FILE.name != "HanVietPreprocessor.py":
    raise ValueError("MODULE_FILE phải trỏ tới HanVietPreprocessor.py")

out_resolved = OUTPUT_DIR.expanduser().resolve()
input_root = Path("/kaggle/input").resolve()
if out_resolved == input_root or input_root in out_resolved.parents:
    raise ValueError("OUTPUT_DIR không được đặt trong /kaggle/input (read-only và là vùng nguồn).")
for name, path in sources.items():
    if path.resolve() == out_resolved or out_resolved in path.resolve().parents:
        raise ValueError(f"{name} đang nằm trong OUTPUT_DIR. Hãy tách Input/Output.")

source_hashes = {}
for lang, src in [("Hán", HAN_FILE), ("Việt", VI_FILE)]:
    raw = src.read_bytes()
    raw.decode("utf-8")  # UTF-8 BOM vẫn đọc được, module sẽ log khi loại bỏ
    if not raw.strip():
        raise ValueError(f"File {lang} trống: {src}")
    source_hashes[lang] = hashlib.sha256(raw).hexdigest()
    print(f"OK {lang}: {src.name} | {len(raw):,} bytes | SHA256 {source_hashes[lang][:16]}...")

if MANUAL_CONFIG_FILE is not None:
    json.loads(Path(MANUAL_CONFIG_FILE).read_text(encoding="utf-8"))
    print("OK cấu hình sửa thủ công")

print("Kiểm tra đường dẫn thành công. Không file nào bị sao chép.")

## Cell 4 — Import module trực tiếp từ input

Không `cp`, không `shutil.copy`, không giải nén ZIP. Cấu hình và văn bản nguồn luôn được đọc từ đường dẫn gốc.

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("hanviet_preprocessor_runtime", MODULE_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f"Không nạp được module: {MODULE_FILE}")
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
HanVietPreprocessor = module.HanVietPreprocessor
print("Đã import:", module.__file__)
print("Không tạo bản sao của module trong OUTPUT_DIR.")

## Cell 5 — Chạy tiền xử lý

Đây là cell **đầu tiên ghi file** và chỉ ghi vào `OUTPUT_DIR`. Nếu bạn đặt `*_START/END`, phần ngoài tiểu mục chỉ bị loại khỏi **bản kết quả chọn vùng**; file TXT nguồn không thay đổi và log có sự kiện `SECTION`.

In [ ]:
processor = HanVietPreprocessor()
result = processor.preprocess(
    han_file=HAN_FILE,
    vi_file=VI_FILE,
    output_dir=OUTPUT_DIR,
    config_path=MANUAL_CONFIG_FILE,
    han_start=HAN_START,
    han_end=HAN_END,
    vi_start=VI_START,
    vi_end=VI_END,
)

print("THÀNH CÔNG — chỉ xuất kết quả tiền xử lý vào:", result.output_dir)
print("Nhật ký mọi thay đổi / vị trí tách:", result.audit_path)
print(json.dumps(result.summary, ensure_ascii=False, indent=2))

## Cell 6 — Xác minh nguyên trạng Input & liệt kê kết quả

Kiểm tra SHA-256 của hai nguồn **trước/sau** để bảo đảm không ghi đè; chỉ liệt kê các file do module tạo, **không chép Input vào Output**.

In [ ]:
expected = {
    "preprocess_changes.txt",
    "preprocess_summary.json",
    "han_clean.txt", "vi_clean.txt",
    "han_sentences.txt", "vi_sentences.txt",
    "han_sentences.jsonl", "vi_sentences.jsonl",
    "han_review_candidates.jsonl", "vi_review_candidates.jsonl",
}
for name, src in [("Hán", HAN_FILE), ("Việt", VI_FILE)]:
    digest = hashlib.sha256(src.read_bytes()).hexdigest()
    assert digest == source_hashes[name], f"Input {name} đã thay đổi!"
print("OK: cả hai file TXT input giữ nguyên (SHA-256 trùng trước/sau).")

actual = {p.name for p in OUTPUT_DIR.iterdir() if p.is_file()}
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(f" - {file.name:35} {file.stat().st_size:>10,} bytes")
    elif file.is_dir():
        print(f" - {file.name}/  (thư mục bổ sung)")

unexpected = actual - expected
if unexpected:
    print("CẢNH BÁO: thư mục output có file từ lần chạy khác:", sorted(unexpected))
else:
    print("OK: OUTPUT_DIR chỉ có file kết quả đã quy định (không có bản sao file input/module).")

## Cell 7 — Xem câu tách, câu cần rà soát và log 4+4

`*_sentences.txt` là ứng viên câu để đưa sang Bertalign / Vecalign / API **sau khi kiểm duyệt**. `needs_review=True` không có nghĩa câu sai; chỉ là thuật toán chưa đủ chứng cứ để tự xác nhận ranh giới.

In [ ]:
for lang, label in [("han", "HÁN"), ("vi", "VIỆT")]:
    rows = result.units[lang]
    print("\n===", label, "===")
    print("Số đơn vị:", len(rows), "| Chờ kiểm tra:", sum(bool(r["needs_review"]) for r in rows))
    for row in rows[:12]:
        flag = " [REVIEW]" if row["needs_review"] else ""
        print(f"{row['id']:>3}. ({row['split_basis']}){flag} {row['text'][:180]}")
    review_file = OUTPUT_DIR / f"{lang}_review_candidates.jsonl"
    candidates = [json.loads(line) for line in review_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("Dòng nghi vấn OCR/chú thích:", len(candidates))
    for x in candidates[:3]:
        print("  -", x)

print("\n=== MẪU LOG SPLIT (4 ký tự trái + 4 ký tự phải) ===")
log_lines = result.audit_path.read_text(encoding="utf-8").splitlines()
splits_shown = 0
for i, line in enumerate(log_lines):
    if "][SPLIT]" in line.replace(" ", "") or "] [SPLIT]" in line:
        print("\n".join(log_lines[i:i+7]))
        print()
        splits_shown += 1
        if splits_shown >= 6:
            break
print("Log đầy đủ:", result.audit_path)

## Cell 8 — Kiểm tra các mốc tách câu thực sự có đủ ngữ cảnh

Mọi bản ghi `SPLIT` trong `preprocess_changes.txt` phải có `raw_left4` và `raw_right4`, mỗi bên **tối đa 4 ký tự Unicode** (có thể ít hơn ở mép văn bản). Chữ trắng / xuống dòng được biểu diễn bằng ký hiệu escaped trong log.

In [ ]:
import re

audit_txt = result.audit_path.read_text(encoding="utf-8")
split_blocks = [block for block in re.split(r"(?=\[\d{6}\])", audit_txt)
                if "][SPLIT]" in block.replace(" ", "") or "] [SPLIT]" in block]

for block in split_blocks:
    fields = {}
    for line in block.splitlines():
        if line.startswith("  ") and ": " in line:
            key, val = line.strip().split(": ", 1)
            fields[key] = json.loads(val)
    assert "raw_left4" in fields and "raw_right4" in fields, "Thiếu ngữ cảnh 4+4"
    assert len(fields["raw_left4"]) <= 4 and len(fields["raw_right4"]) <= 4, "Vượt quá 4 ký tự"
print(f"OK: đã xác minh {len(split_blocks)} sự kiện SPLIT, có ngữ cảnh 4+4.")
print("Xem báo cáo đầy đủ:", OUTPUT_DIR / "preprocess_summary.json")
print("Đầu ra cho bước dóng hàng:", OUTPUT_DIR / "han_sentences.txt", "và", OUTPUT_DIR / "vi_sentences.txt")

## Output cuối cùng trong `/kaggle/working/nu_khoa_preprocessed`

```text
nu_khoa_preprocessed/
├── preprocess_changes.txt          # Mọi chỉnh sửa + mọi vị trí SPLIT với 4 ký tự mỗi bên
├── preprocess_summary.json         # Thống kê, SHA-256 nguồn
├── han_clean.txt                   # Kết quả sạch hóa từ bản Hán (KHÔNG phải copy bản gốc)
├── vi_clean.txt                    # Kết quả sạch hóa từ bản Việt
├── han_sentences.txt               # Một ứng viên câu trên một dòng
├── vi_sentences.txt
├── han_sentences.jsonl             # Kèm offset và needs_review
├── vi_sentences.jsonl
├── han_review_candidates.jsonl     # Dòng đáng ngờ, chưa tự xóa
└── vi_review_candidates.jsonl
```

Trong Kaggle chọn **Save Version → Output** hoặc xem tab **Output** để lấy kết quả. Không cần upload lại input và không cần tạo ZIP chứa dữ liệu gốc.

**Lưu ý về kiểm duyệt:** Ranh giới câu chỉ là ứng viên; trước bước voting Bertalign/Vecalign/API, hãy kiểm tra tiểu mục Hán–Việt thực sự trùng nhau, giữ bản gốc và rà các dòng `REVIEW`. Khi không tìm thấy bản dịch, không được tự xóa dữ liệu.